In [ ]:
!pip uninstall -y datasets
!pip install "datasets<4.0.0"

Load Dataset

In [ ]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
import torch
from torch.utils.data import Dataset, DataLoader
import re
import torch.nn as nn
import os
import matplotlib.pyplot as plt

dataset = load_dataset("CogComp/trec", trust_remote_code=True)

Split Training / Validation Set and Test Set

In [ ]:
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

def make_balanced_subset(df, n_total, label_col="coarse_label", seed=42):
    n_classes = df[label_col].nunique()
    n_per_class = n_total // n_classes

    subset = (
        df.groupby(label_col, group_keys=False)
          .apply(lambda x: x.sample(n=min(len(x), n_per_class), random_state=seed))
          .sample(frac=1, random_state=seed)
          .reset_index(drop=True)
    )

    return subset

train_100 = make_balanced_subset(train_df, 100)
train_500 = make_balanced_subset(train_df, 500)
train_1000 = make_balanced_subset(train_df, 1000)
train_full = train_df

train_subsets = {
    "100": train_100,
    "500": train_500,
    "1000": train_1000,
    "full": train_full,
}


In [ ]:
# check
print(train_df["coarse_label"].value_counts().sort_index())

for size_name, subset_df in train_subsets.items():
    X_train = subset_df["text"]
    y_train = subset_df["coarse_label"]

    print(size_name, len(X_train))

In [ ]:
def split_train_val(df, label_col="coarse_label", val_size=0.2, seed=42):
    train_part, val_part = train_test_split(
        df,
        test_size=val_size,
        random_state=seed,
        stratify=df[label_col]
    )
    return train_part.reset_index(drop=True), val_part.reset_index(drop=True)

splits = {}

for name, subset in train_subsets.items():
    train_part, val_part = split_train_val(subset)

    splits[name] = {
        "train": train_part,
        "val": val_part,
        "test": test_df
    }

    print(name)
    print("train:", len(train_part))
    print("val:", len(val_part))
    print("test:", len(test_df))

Multinomial Naive Bayes

In [ ]:
def evaluate_model(model, X, y):
    preds = model.predict(X)
    probs = model.predict_proba(X)

    acc = accuracy_score(y, preds)
    macro_f1 = f1_score(y, preds, average="macro")

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "probs": probs,
        "preds": preds
    }

def expected_calibration_error(y_true, probs, n_bins=10):
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    correctness = (predictions == np.array(y_true))

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        bin_lower = bin_edges[i]
        bin_upper = bin_edges[i + 1]

        in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
        prop_in_bin = np.mean(in_bin)

        if prop_in_bin > 0:
            accuracy_in_bin = np.mean(correctness[in_bin])
            confidence_in_bin = np.mean(confidences[in_bin])
            ece += prop_in_bin * abs(accuracy_in_bin - confidence_in_bin)

    return ece

calibration_store = {}

In [ ]:
alphas = [0.1, 0.5, 1.0, 2.0]

nb_results = []

for size_name, data in splits.items():
    X_train = data["train"]["text"]
    y_train = data["train"]["coarse_label"]

    X_val = data["val"]["text"]
    y_val = data["val"]["coarse_label"]

    X_test = data["test"]["text"]
    y_test = data["test"]["coarse_label"]

    best_alpha = None
    best_val_f1 = -1
    best_model = None

    for alpha in alphas:
        model = Pipeline([
            ("vectorizer", CountVectorizer(lowercase=True)),
            ("classifier", MultinomialNB(alpha=alpha))
        ])

        model.fit(X_train, y_train)

        val_eval = evaluate_model(model, X_val, y_val)

        if val_eval["macro_f1"] > best_val_f1:
            best_val_f1 = val_eval["macro_f1"]
            best_alpha = alpha
            best_model = model

    test_eval = evaluate_model(best_model, X_test, y_test)
    test_ece = expected_calibration_error(y_test, test_eval["probs"])
    if size_name in ["500", "full"]:
        calibration_store[("MultinomialNB", size_name)] = {
            "y_true": np.array(y_test),
            "probs": test_eval["probs"],
        }

    nb_results.append({
        "model": "MultinomialNB",
        "train_size": size_name,
        "best_alpha": best_alpha,
        "val_macro_f1": best_val_f1,
        "test_accuracy": test_eval["accuracy"],
        "test_macro_f1": test_eval["macro_f1"],
        "test_ece": test_ece
    })

nb_results

In [ ]:
# Creat plot set
nb_results_df = pd.DataFrame(nb_results)
nb_results_df

Logistic Regression

In [ ]:
Cs = [0.1, 1.0, 10.0]

lr_results = []

for size_name, data in splits.items():
    X_train = data["train"]["text"]
    y_train = data["train"]["coarse_label"]

    X_val = data["val"]["text"]
    y_val = data["val"]["coarse_label"]

    X_test = data["test"]["text"]
    y_test = data["test"]["coarse_label"]

    best_C = None
    best_val_f1 = -1
    best_model = None

    for C in Cs:
        model = Pipeline([
            ("vectorizer", TfidfVectorizer(lowercase=True)),
            ("classifier", LogisticRegression(
                C=C,
                max_iter=1000,
                solver="lbfgs",
                multi_class="auto"
            ))
        ])

        model.fit(X_train, y_train)

        val_eval = evaluate_model(model, X_val, y_val)

        if val_eval["macro_f1"] > best_val_f1:
            best_val_f1 = val_eval["macro_f1"]
            best_C = C
            best_model = model

    test_eval = evaluate_model(best_model, X_test, y_test)
    test_ece = expected_calibration_error(y_test, test_eval["probs"])
    if size_name in ["500", "full"]:
        calibration_store[("LogisticRegression", size_name)] = {
            "y_true": np.array(y_test),
            "probs": test_eval["probs"],
        }

    lr_results.append({
        "model": "LogisticRegression",
        "train_size": size_name,
        "best_C": best_C,
        "val_macro_f1": best_val_f1,
        "test_accuracy": test_eval["accuracy"],
        "test_macro_f1": test_eval["macro_f1"],
        "test_ece": test_ece
    })

lr_results_df = pd.DataFrame(lr_results)
lr_results_df


In [ ]:
# plot set
classical_results_df = pd.concat(
    [nb_results_df, lr_results_df],
    ignore_index=True
)

classical_results_df

BiLSTM


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# Word-level Tokenizer & Vocabulary
def simple_tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

def build_vocab(texts, min_freq=1):
    counter = Counter()
    for text in texts:
        counter.update(simple_tokenize(text))

    vocab = {"<pad>": 0, "<unk>": 1}
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)

    return vocab

In [ ]:
class TRECDataset(Dataset):
    def __init__(self, df, vocab, max_len=30):
        self.texts = df["text"].tolist()
        self.labels = df["coarse_label"].tolist()
        self.vocab = vocab
        self.max_len = max_len

    def encode_text(self, text):
        tokens = simple_tokenize(text)
        ids = [self.vocab.get(tok, self.vocab["<unk>"]) for tok in tokens]

        if len(ids) < self.max_len:
            ids = ids + [self.vocab["<pad>"]] * (self.max_len - len(ids))
        else:
            ids = ids[:self.max_len]

        return ids

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.encode_text(self.texts[idx]), dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate_torch_model(model, loader, device):
    model.eval()

    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids)
            probs = torch.softmax(logits, dim=1)

            preds = torch.argmax(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "probs": np.array(all_probs),
        "preds": np.array(all_preds),
        "labels": np.array(all_labels)
    }

GloVe-BiLSTM

In [ ]:
if not os.path.exists("glove.6B.100d.txt"):
    !wget -q http://nlp.stanford.edu/data/glove.6B.zip
    !unzip -oq glove.6B.zip

In [ ]:
def load_glove_embeddings(glove_path):
    glove = {}

    with open(glove_path, "r", encoding="utf-8") as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype="float32")
            glove[word] = vector

    return glove

glove = load_glove_embeddings("glove.6B.100d.txt")
print(len(glove))

In [ ]:
def build_embedding_matrix(vocab, glove, embed_dim=100):
    embedding_matrix = np.random.normal(0, 0.1, (len(vocab), embed_dim)).astype("float32")

    embedding_matrix[vocab["<pad>"]] = np.zeros(embed_dim)

    found = 0
    for word, idx in vocab.items():
        if word in glove:
            embedding_matrix[idx] = glove[word]
            found += 1

    print(f"Found {found}/{len(vocab)} words in GloVe")

    return torch.tensor(embedding_matrix, dtype=torch.float)

In [ ]:
class GloveBiLSTMClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_size=128, num_classes=6, dropout=0.3, freeze_embeddings=False):
        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix,
            freeze=freeze_embeddings,
            padding_idx=0
        )

        embed_dim = embedding_matrix.shape[1]

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        _, (hidden, _) = self.lstm(x)

        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]
        h = torch.cat([forward_hidden, backward_hidden], dim=1)

        h = self.dropout(h)
        logits = self.fc(h)

        return logits

In [ ]:
glove_bilstm_results = []

batch_size = 32
epochs = 10
learning_rates = [1e-3]
hidden_sizes = [128]
dropouts = [0.3]

for size_name, data in splits.items():
    vocab = build_vocab(data["train"]["text"])
    embedding_matrix = build_embedding_matrix(vocab, glove, embed_dim=100)

    train_dataset = TRECDataset(data["train"], vocab)
    val_dataset = TRECDataset(data["val"], vocab)
    test_dataset = TRECDataset(data["test"], vocab)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)

    best_val_f1 = -1
    best_config = None
    best_state = None

    for lr in learning_rates:
        for hidden_size in hidden_sizes:
            for dropout in dropouts:
                model = GloveBiLSTMClassifier(
                    embedding_matrix=embedding_matrix,
                    hidden_size=hidden_size,
                    dropout=dropout,
                    freeze_embeddings=False
                ).to(device)

                optimizer = torch.optim.Adam(model.parameters(), lr=lr)
                criterion = nn.CrossEntropyLoss()

                for epoch in range(epochs):
                    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
                    val_eval = evaluate_torch_model(model, val_loader, device)

                if val_eval["macro_f1"] > best_val_f1:
                    best_val_f1 = val_eval["macro_f1"]
                    best_config = {
                        "lr": lr,
                        "hidden_size": hidden_size,
                        "dropout": dropout,
                        "freeze_embeddings": False
                    }
                    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    best_model = GloveBiLSTMClassifier(
        embedding_matrix=embedding_matrix,
        hidden_size=best_config["hidden_size"],
        dropout=best_config["dropout"],
        freeze_embeddings=best_config["freeze_embeddings"]
    ).to(device)

    best_model.load_state_dict(best_state)

    test_eval = evaluate_torch_model(best_model, test_loader, device)
    test_ece = expected_calibration_error(test_eval["labels"], test_eval["probs"])
    if size_name in ["500", "full"]:
        calibration_store[("BiLSTM-GloVe", size_name)] = {
            "y_true": test_eval["labels"],
            "probs": test_eval["probs"],
        }

    glove_bilstm_results.append({
        "model": "BiLSTM-GloVe",
        "train_size": size_name,
        "best_config": best_config,
        "vocab_size": len(vocab),
        "val_macro_f1": best_val_f1,
        "test_accuracy": test_eval["accuracy"],
        "test_macro_f1": test_eval["macro_f1"],
        "test_ece": test_ece
    })

glove_bilstm_results_df = pd.DataFrame(glove_bilstm_results)
glove_bilstm_results_df

In [ ]:
# plot set
all_results_df = pd.concat(
    [classical_results_df, glove_bilstm_results_df],
    ignore_index=True
)

all_results_df


DistilBERT


In [ ]:
!pip install transformers accelerate -q

In [ ]:
import torch
import numpy as np
import pandas as pd

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
# Define Tokenizer & Metrics
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }
# Transform to Hugging Face Dataset
def df_to_hf_dataset(df):
    temp_df = df[["text", "coarse_label"]].copy()
    temp_df = temp_df.rename(columns={"coarse_label": "labels"})
    return Dataset.from_pandas(temp_df, preserve_index=False)

In [ ]:
distilbert_results = []

learning_rates = [2e-5]

for size_name, data in splits.items():
    print(f"\n===== Training DistilBERT on {size_name} =====")

    train_dataset = df_to_hf_dataset(data["train"])
    val_dataset = df_to_hf_dataset(data["val"])
    test_dataset = df_to_hf_dataset(data["test"])

    train_dataset = train_dataset.map(tokenize_function, batched=True)
    val_dataset = val_dataset.map(tokenize_function, batched=True)
    test_dataset = test_dataset.map(tokenize_function, batched=True)

    train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

    best_lr = None
    best_val_f1 = -1
    best_trainer = None

    for lr in learning_rates:
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=6
        )

        training_args = TrainingArguments(
            output_dir=f"./distilbert_trec_{size_name}_{lr}",
            eval_strategy="epoch",
            save_strategy="epoch",
            learning_rate=lr,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=32,
            num_train_epochs=4,
            weight_decay=0.01,
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            greater_is_better=True,
            logging_steps=20,
            report_to="none"
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics
        )

        trainer.train()

        val_metrics = trainer.evaluate(val_dataset)
        val_f1 = val_metrics["eval_macro_f1"]

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_lr = lr
            best_trainer = trainer

    test_metrics = best_trainer.evaluate(test_dataset)

    predictions = best_trainer.predict(test_dataset)
    logits = predictions.predictions
    labels = predictions.label_ids

    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    test_ece = expected_calibration_error(labels, probs)
    if size_name in ["500", "full"]:
        calibration_store[("DistilBERT", size_name)] = {
            "y_true": labels,
            "probs": probs,
        }

    distilbert_results.append({
        "model": "DistilBERT",
        "train_size": size_name,
        "best_lr": best_lr,
        "val_macro_f1": best_val_f1,
        "test_accuracy": test_metrics["eval_accuracy"],
        "test_macro_f1": test_metrics["eval_macro_f1"],
        "test_ece": test_ece
    })

distilbert_results_df = pd.DataFrame(distilbert_results)
distilbert_results_df

In [ ]:
# plot set
final_results_df = pd.concat(
    [classical_results_df, glove_bilstm_results_df, distilbert_results_df],
    ignore_index=True
)

final_results_df

In [ ]:
# plot F1
import matplotlib.pyplot as plt

plot_df = final_results_df.copy()
plot_df["train_size_numeric"] = plot_df["train_size"].map({
    "100": 100,
    "500": 500,
    "1000": 1000,
    "full": 5452
})

plt.figure(figsize=(8, 5))

for model_name in plot_df["model"].unique():
    model_df = plot_df[plot_df["model"] == model_name]
    plt.plot(
        model_df["train_size_numeric"],
        model_df["test_macro_f1"],
        marker="o",
        label=model_name
    )

plt.xlabel("Training set size")
plt.ylabel("Test Macro-F1")
plt.title("Test Macro-F1 vs Training Set Size")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# plot ECE
plt.figure(figsize=(8, 5))

for model_name in plot_df["model"].unique():
    model_df = plot_df[plot_df["model"] == model_name]
    plt.plot(
        model_df["train_size_numeric"],
        model_df["test_ece"],
        marker="o",
        label=model_name
    )

plt.xlabel("Training set size")
plt.ylabel("Expected Calibration Error")
plt.title("ECE vs Training Set Size")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
def reliability_points(y_true, probs, n_bins=10):
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    correctness = (predictions == np.array(y_true))

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)

    accs, confs = [], []
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        in_bin = (confidences > lo) & (confidences <= hi)
        if np.sum(in_bin) > 0:
            accs.append(np.mean(correctness[in_bin]))
            confs.append(np.mean(confidences[in_bin]))
        else:
            accs.append(np.nan)
            confs.append(np.nan)

    return np.array(confs), np.array(accs)

models = ["MultinomialNB", "LogisticRegression", "BiLSTM-GloVe", "DistilBERT"]
sizes = ["500", "full"]

# 4 rows (models) x 2 cols (train sizes): much larger each subplot
fig, axes = plt.subplots(4, 2, figsize=(13, 18), sharex=True, sharey=True)

for r, model_name in enumerate(models):
    for c, size_name in enumerate(sizes):
        ax = axes[r, c]
        key = (model_name, size_name)

        if key not in calibration_store:
            ax.text(0.5, 0.5, "Missing data", ha="center", va="center")
            ax.set_title(f"{model_name} | train={size_name}", fontsize=11)
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            continue

        y_true = calibration_store[key]["y_true"]
        probs = calibration_store[key]["probs"]

        confs, accs = reliability_points(y_true, probs, n_bins=10)
        mask = ~np.isnan(accs)

        ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
        ax.plot(confs[mask], accs[mask], marker="o", linewidth=2, markersize=5)

        ax.set_title(f"{model_name} | train={size_name}", fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)

        if c == 0:
            ax.set_ylabel("Empirical Accuracy", fontsize=10)
        if r == len(models) - 1:
            ax.set_xlabel("Mean Predicted Confidence", fontsize=10)

plt.suptitle("Reliability Diagrams (4 models x 2 train sizes: 500/full)", y=0.995, fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.985])
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

label_names = ["ABBR", "DESC", "ENTY", "HUM", "LOC", "NUM"]

def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(6)))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)

    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, cmap="Blues", values_format="d")
    plt.title(title)
    plt.show()

In [ ]:
# rebuild full test dataset
full_test_dataset = df_to_hf_dataset(splits["full"]["test"])
full_test_dataset = full_test_dataset.map(tokenize_function, batched=True)
full_test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# use the last/best trainer if it is still in memory
predictions = best_trainer.predict(full_test_dataset)

logits = predictions.predictions
labels = predictions.label_ids
probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
preds = np.argmax(logits, axis=1)

print(labels[:10])
print(preds[:10])

In [ ]:
plot_confusion_matrix(labels, preds, "DistilBERT Full Confusion Matrix")

In [ ]:
def get_mistakes(test_df, y_true, y_pred, probs, label_names, n=10):
    mistakes = []

    for i in range(len(y_true)):
        if y_true[i] != y_pred[i]:
            mistakes.append({
                "text": test_df.iloc[i]["text"],
                "true_label": label_names[y_true[i]],
                "pred_label": label_names[y_pred[i]],
                "confidence": float(np.max(probs[i]))
            })

    mistakes_df = pd.DataFrame(mistakes)
    return mistakes_df.sort_values("confidence", ascending=False).head(n)

mistakes_df = get_mistakes(
    test_df,
    labels,
    np.argmax(logits, axis=1),
    probs,
    label_names,
    n=10
)

mistakes_df